# TechJobAI — Demo A→Z

Chạy toàn bộ pipeline từ raw Kaggle dataset → trained models → server hoạt động.

## Pipeline
```
Raw CSVs (Kaggle) ──► preprocess_kaggle.py ──► it_jobs_processed.csv ──► retrain_all.py ──► models/*.joblib ──► backend.server (Flask)
```

## Yêu cầu
- Python 3.10+
- `pip install -r requirements.txt` đã chạy
- File `Dataset/linkedin_job_postings.csv`, `job_skills.csv`, `job_summary.csv` tồn tại

In [ ]:
import os, sys, json, subprocess, time, threading, requests
import pandas as pd
import numpy as np

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(BASE)
sys.path.insert(0, BASE)

print(f"Working dir: {BASE}")
print(f"Python: {sys.version}")

---
## 1. Kiểm tra raw data tồn tại

In [ ]:
required_files = [
    'Dataset/linkedin_job_postings.csv',
    'Dataset/job_skills.csv',
    'Dataset/job_summary.csv',
]
for f in required_files:
    ok = os.path.exists(f)
    print(f"  {'OK' if ok else 'MISSING'} {f}")
assert all(os.path.exists(f) for f in required_files), "Thiếu raw dataset"
print("\nAll raw files present.")

---
## 2. Preprocessing — chạy `preprocess_kaggle.py`

Đọc 3 CSV gốc → lọc IT jobs → trích seniority, skills, domain → `data/it_jobs_processed.csv`

Khoảng 3-5 phút trên máy thường.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("preprocess",
    os.path.join(BASE, 'Dataset', 'preprocess_kaggle.py'))
preprocess = importlib.util.module_from_spec(spec)

print("Running preprocess_kaggle...")
start = time.time()
spec.loader.exec_module(preprocess)
elapsed = time.time() - start
print(f"\nPreprocess done in {elapsed/60:.1f} min")

In [ ]:
df = pd.read_csv('data/it_jobs_processed.csv')
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"IT domains: {sorted(df['it_domain'].dropna().unique())}")
print(f"Seniority levels: {sorted(df['seniority_level'].dropna().unique())}")
print(f"States: {sorted(df['state'].dropna().unique())}")
df.head(3)

---
## 3. Train models — chạy `retrain_all.py`

Train 3 models từ `it_jobs_processed.csv`:
- Salary Predictor (XGBoost / tuned RF, R²~0.53)
- Demand Scorer (RandomForest, R²~0.67)
- Cluster (KMeans + PCA, K=5)

Khoảng 10-20 phút.

In [ ]:
print("Running retrain_all...")
start = time.time()

result = subprocess.run(
    [sys.executable, 'retrain_all.py'],
    capture_output=True, text=True, cwd=BASE
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

elapsed = time.time() - start
print(f"\nTraining done in {elapsed/60:.1f} min")

In [ ]:
model_files = [f for f in os.listdir('models') if f.endswith('.joblib')]
print("Model files created:")
for f in sorted(model_files):
    size_mb = os.path.getsize(os.path.join('models', f)) / 1e6
    print(f"  {f:40s} {size_mb:.1f} MB")

---
## 4. Validate models

Kiểm tra model metrics từ metadata files.

In [ ]:
import joblib

meta_sal = joblib.load('models/salary_model_meta.joblib')
meta_dem = joblib.load('models/demand_meta.joblib')
meta_cl = joblib.load('models/cluster_meta.joblib')

print("=== SALARY MODEL ===")
print(f"  R²:  {meta_sal['r2_score']:.4f}")
print(f"  MAE: ${meta_sal['mae']:,.0f}")
print(f"  Train: {meta_sal['train_size']:,} | Test: {meta_sal['test_size']:,}")

print("\n=== DEMAND MODEL ===")
print(f"  R²: {meta_dem['r2_score']:.4f}")

print("\n=== CLUSTER MODEL ===")
print(f"  Silhouette: {meta_cl['silhouette_score']:.4f}")
for c, desc in sorted(meta_cl['cluster_descriptions'].items()):
    print(f"  {desc}")

print(f"\nDomains: {meta_sal['it_domain']}")
print(f"Seniority: {meta_sal['seniority_level']}")

---
## 5. Start Flask server

Server chạy trên port 5000. Dừng bằng nút **Interrupt kernel** (□) hoặc run cell cuối.

In [ ]:
from backend.server import app

def run_server():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("Server started at http://localhost:5000")

---
## 6. Test API endpoints

Gọi các API chính để verify server hoạt động.

In [ ]:
def api(method='GET', path='/api/health', **kwargs):
    url = f"http://localhost:5000{path}"
    r = requests.request(method, url, **kwargs, timeout=10)
    print(f"{r.status_code} {method} {path}")
    return r.json()

# Health
print("=== Health ===")
h = api('GET', '/api/health')
print(json.dumps(h, indent=2))

In [ ]:
# Predict salary
print("=== Predict ===")
pred = api('POST', '/predict', json={
    'job_title': 'Software Engineer',
    'seniority_level': 'Senior',
    'state': 'CA',
    'skills': ['Python', 'SQL', 'AWS']
})
print(json.dumps(pred, indent=2))

In [ ]:
# Cluster
print("=== Cluster ===")
cl = api('POST', '/cluster', json={
    'job_title': 'Data Scientist',
    'seniority_level': 'Mid',
    'state': 'NY',
    'skills': ['Python', 'TensorFlow', 'SQL']
})
print(json.dumps(cl, indent=2))

In [ ]:
# Demand score
print("=== Demand ===")
dem = api('POST', '/demand', json={
    'job_title': 'DevOps Engineer',
    'state': 'TX',
    'seniority_level': 'Senior',
    'it_domain': 'DevOps'
})
print(json.dumps(dem, indent=2))

In [ ]:
# Meta (model info)
print("=== Meta ===")
meta = api('GET', '/api/meta')
print(f"Salary R²: {meta.get('salary_meta', {}).get('r2_score', 'N/A')}")
print(f"Demand R²: {meta.get('demand_meta', {}).get('r2_score', 'N/A')}")
print(f"Cluster silhouette: {meta.get('cluster_meta', {}).get('silhouette_score', 'N/A')}")
print(f"Clusters: {list(meta.get('cluster_meta', {}).get('cluster_descriptions', {}).values())}")

In [ ]:
# Charts data (top domains, seniority distribution)
print("=== Charts ===")
charts = api('GET', '/api/charts')
print(f"Top domains: {len(charts.get('domain_counts', {}))} entries")
print(f"Seniority: {charts.get('seniority_distribution', {})}")
print(f"States: {len(charts.get('state_counts', {}))} entries")
print(f"Sample job titles: {charts.get('top_titles', [])[:5]}")

In [ ]:
# Realtime trends (free API, needs internet)
print("=== Realtime Trends ===")
try:
    rt = api('POST', '/realtime/trends', json={'location': 'New York'})
    print(f"Source: {rt.get('source', 'N/A')}")
    print(f"Jobs fetched: {rt.get('total_jobs', 0)}")
    print(f"Top skills: {[s['name'] for s in rt.get('top_skills', [])[:5]]}")
    print(f"Top locations: {[s['name'] for s in rt.get('top_locations', [])[:5]]}")
except Exception as e:
    print(f"Skipped (needs internet): {e}")

In [ ]:
# CV Parser (optional)
print("=== CV Parser (no file → default) ===")
cv = api('POST', '/predict/cv-parse', json={})
print(f"Extracted title: {cv.get('job_title', 'N/A')}")
print(f"Domain: {cv.get('it_domain', 'N/A')}")
print(f"State: {cv.get('state', 'N/A')}")

---
## 7. Dashboard (HTML)

Mở trình duyệt: [http://localhost:5000](http://localhost:5000)

---
## 8. Dọn dẹp — tắt server

Chạy cell dưới để shutdown server. Nếu không được, ấn **Interrupt (□)** trên toolbar.

In [ ]:
import os
os._exit(0)  # Force kill kernel (including Flask thread)